# Part 1: Build knowledge bases with agentic retrieval

Cocoarynth Chocolate Works is a small bean-to-bar chocolate maker in Portland, Oregon. It buys cacao beans from farms in five countries, makes single-origin bars, and sells them through grocers, specialty shops, cafés, and an overseas distributor. Its records are scattered: farm logs, factory test results, shipping paperwork, retailer emails, and company policies. Some are in Spanish, and some facts only appear in charts, photos, or scanned handwritten forms.

Customers and retailers keep asking questions that no single document can answer. You'll build knowledge bases over Cocoarynth's records and use them to solve those mysteries.

**Mission**
- Upload Cocoarynth's label and marketing claims policy to a **file knowledge source** and find out when a bar may carry the organic seal.
- Connect four indexes of Cocoarynth records as **indexed knowledge sources**: origins, quality, logistics and retail, and policies.
- Build a four-source **knowledge base** and use it to solve mysteries whose answers are spread across several documents.

## The building blocks

In Foundry IQ (Azure AI Search), a **knowledge base** connects a chat completion model to one or more knowledge sources. It defines which sources to query, which model to use for reasoning, and how retrieval should work.

In this notebook, you'll explore two types of knowledge sources:

1. **File knowledge source**: Upload raw files directly. The service handles chunking, embedding, and indexing automatically.
2. **Indexed knowledge source**: Connect to a pre-built search index with control over schema and processing.

The label claims policy is also included in the policies index. The standalone policy KB demonstrates file ingestion; the combined KB uses only the four indexed sources.

## Pre-configured environment

This lab uses the following Azure resources:

* Foundry IQ (Azure AI Search), serverless in West Central US
  * `cocoarynth-origins`: 16 farm and supplier records, including collection tickets, fermentation and drying logs, inspections, certificates, and sourcing prices.
  * `cocoarynth-quality`: 15 factory records, including bean tests, holds, lab certificates, tasting panels, batch records, and nonconformance reports.
  * `cocoarynth-logistics-retail`: 16 shipping and sales records, including bills of lading, carrier logs, retailer emails, agreements, launch plans, and sales.
  * `cocoarynth-policies`: 8 company policies, guides, and reference documents.
  * Semantic ranking and 3072-dimensional embeddings.
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-5.4`: Reasoning and answer synthesis.
  * `text-embedding-3-large`: Vector embeddings.

All four indexes must be populated before you start. Provisioning restores them from snapshots captured from a live ingestion run. Chunk counts depend on ingestion and are not fixed PDF counts.

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Foundry IQ (Azure AI Search) endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

REPO_ROOT = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / ".env.sample").is_file() and (directory / "infra").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the LAB673 repository.")
if not (REPO_ROOT / ".env").is_file():
    raise RuntimeError("Provision the lab environment before running this notebook.")
load_dotenv(REPO_ROOT / ".env", override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

ORIGINS_INDEX = "cocoarynth-origins"
QUALITY_INDEX = "cocoarynth-quality"
LOGISTICS_RETAIL_INDEX = "cocoarynth-logistics-retail"
POLICIES_INDEX = "cocoarynth-policies"
INDEX_NAMES = [ORIGINS_INDEX, QUALITY_INDEX, LOGISTICS_RETAIL_INDEX, POLICIES_INDEX]

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

---

## A: File Knowledge Source

A **File Knowledge Source** lets you upload raw files directly. The service handles chunking, embedding, and indexing automatically. This is the fastest way to get documents into a knowledge base.

## Step 2: Create a File Knowledge Source

Configure a file knowledge source with an embedding model for automatic ingestion. The `ingestion_parameters` specify:
- **Embedding model**: Generates vector embeddings for the uploaded content
- **Content extraction mode**: How to extract text from documents

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "cocoarynth-policy-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Cocoarynth label and marketing claims policy defining when wrappers may carry origin, farm, organic, and other claims",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## Step 3: Upload files to the knowledge source

Upload the label and marketing claims policy directly to the file knowledge source. The service extracts text, chunks the document, and generates embeddings.

Ingestion time varies. If the first retrieval reports that the file is not ready, allow ingestion to finish and rerun the retrieval cell.


In [ ]:
data_dir = REPO_ROOT / "data"
files_to_upload = [
    data_dir / "cocoarynth" / "documents" / POLICIES_INDEX / "label-claims-policy.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as pdf_file:
        file_content = pdf_file.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: Create a knowledge base for the file source

A knowledge base is the orchestration layer that combines:

1. Your data sources (the file knowledge source)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the attached LLM will generate a complete natural language answer from the retrieved chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "cocoarynth-policy-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    description="Cocoarynth label and marketing claims policy knowledge base",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 5: Query the file knowledge base

Ask a question about the uploaded policy. The knowledge base will search across the file content and synthesize a natural language answer with citations.

Remember the answer: you'll need it for one of the mysteries in Part B.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "When may a Cocoarynth bar carry the organic seal, and what if the supplier's organic certificate is pending renewal?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

The activity log shows the sequence of steps taken by the knowledge base. Each activity contains:

* `type`: The type of activity. For this log, you will see the first step of "modelQueryPlanning", multiple "file" steps (one for each query sent to the file knowledge source), "modelAnswerSynthesis" for the answer synthesis step, and "agenticReasoning" for the semantic re-ranking step.
* `elapsedMS`: The duration of the activity.
* `id`: The activity ID, which is helpful to connect each reference to the activity that produced it. You'll review references next.

The activity entries also contain other metadata specific to their type, so that you can see how many tokens were used by any model-powered steps and see the queries sent to sources for retrieval steps.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references show which sources were retrieved, and the source metadata for each reference. Every result contains:

* `type`: Always "file", since this knowledge base only contains a file source.
* `id`: A numeric identifier for the reference. This is what appears in the answer as citation markers like `[ref:1]`.
* `activitySource`: The id of the retrieval activity in the activity log that produced this reference. You can use this to trace which search step returned this document.
* `sourceData`: Fields from the retrieved chunk in the auto-created search index - always `uid`, `snippet`, and `metadata_storage_path` for file source indexes.
* `rerankerScore`: A score between 0-4 from the re-ranking model, where 4 is the best.
* `docName`: The auto-generated filename for the uploaded file.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: Indexed knowledge sources

**Indexed knowledge sources** connect to pre-built search indexes, giving you control over schema, chunking, and embeddings.

Cocoarynth's records are split across four indexes, the way they'd be split across teams: the sourcing team's farm and supplier records, the factory's quality records, the logistics and sales team's shipping and retailer records, and company-wide policies.

## Step 6: Verify search indexes

Confirm that all four indexes exist and contain chunks. Content Understanding produces multiple chunks per PDF, so the final counts depend on ingestion. An empty index is not ready for the exercises.

In [ ]:
from azure.search.documents import SearchClient

for index_name in INDEX_NAMES:
    client = SearchClient(
        endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
        index_name=index_name,
        credential=search_credential,
    )
    count = client.get_document_count()
    if count == 0:
        client.close()
        raise RuntimeError(f"Index '{index_name}' is empty. Complete document ingestion or snapshot restoration first.")
    print(f"{index_name}: {count} document chunks")
    client.close()

## Step 7: Create four indexed knowledge sources

Create one knowledge source for each populated index. Each description tells the knowledge base what the source contains, which helps it decide where to look:

* `cocoarynth-origins-source`: Farm and supplier records, partly in Spanish.
* `cocoarynth-quality-source`: Factory receiving, testing, tasting, and batch records.
* `cocoarynth-logistics-retail-source`: Shipping, carrier, retailer, and sales records.
* `cocoarynth-policies-source`: Company policies, guides, and reference material.

All sources search the `chunk` field and return `blob_path`, `title`, `page_number_from`, and `page_number_to` as citation metadata. Image extraction is not part of this lab's portable index schema.


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

INDEX_DESCRIPTIONS = {
    ORIGINS_INDEX: "Cocoarynth origin and farm records, partly in Spanish: supplier profiles, collection tickets, fermentation and drying logs, farm inspections, organic certificates, and sourcing prices",
    QUALITY_INDEX: "Cocoarynth factory quality and production records: bean receiving tests, holds, lab certificates, tasting panels, batch records, and nonconformance reports",
    LOGISTICS_RETAIL_INDEX: "Cocoarynth shipping and sales records: bills of lading, carrier and depot logs, retailer emails and complaints, agreements, launch plans, price lists, and sales",
    POLICIES_INDEX: "Cocoarynth company-wide policies, guides, and reference material, including storage, labeling, and data-sharing rules",
}
KNOWLEDGE_SOURCE_NAMES = {index_name: f"{index_name}-source" for index_name in INDEX_NAMES}

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for index_name, knowledge_source_name in KNOWLEDGE_SOURCE_NAMES.items():
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=INDEX_DESCRIPTIONS[index_name],
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")

## Step 8: Create the multi-source knowledge base

Create `cocoarynth-kb-all` with all four indexed sources. The standalone file KB remains a separate exercise; the label claims policy is already present in the policies index.

The knowledge base uses **auto reasoning effort**, so the model decides how much planning each question needs. A simple question gets one round of searches; a mystery whose clues are spread across documents can get follow-up searches based on what the first round found.

This notebook uses `ANSWER_SYNTHESIS` so the model returns a grounded natural-language answer.


In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalAutoReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "cocoarynth-kb-all"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Cocoarynth knowledge base over origin, quality, logistics and retail, and policy records",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES.values()],
    retrieval_reasoning_effort=KnowledgeRetrievalAutoReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 9: Solve a mystery with the multi-source knowledge base

In July 2026, Larkspur Market emailed Cocoarynth photos of Ghana bars that had turned grey and dusty-looking on the shelf. Was the batch bad? Ask the knowledge base:

* _"Why did the Ghana bars sold at Larkspur Market turn grey and dusty-looking in July 2026, and was the batch itself bad?"_

No single document holds the answer: the clues are in a retailer email, a carrier's records, a factory quality check, and a storage guide. Watch which sources the knowledge base searches.

Then try the other mysteries by changing `question` and rerunning the cell:

* _"A Pantry Lane customer sent a photo of an Ecuador bar wrapper and asked which farms grew the cacao. What can we tell them?"_
* _"Why did the trial beans from the Dominican Republic taste smoky, and what happened next?"_
* _"Can Cocoarynth print the organic seal on the Peru Bagua 70% bars packed in September 2026?"_
* _"Why did the Madagascar limited-edition bar with cacao nibs launch on 1 October instead of 1 September, and what happened at the estate to start the delay?"_
* _"Which origin won Cocoarynth's September 2026 tasting panel, and what did the tasters notice about it?"_
* _"Which origin's beans cost Cocoarynth the most in 2026, and was it worth paying for them, judging by taste and sales?"_


In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

knowledge_source_params = [
    SearchIndexKnowledgeSourceParams(
        knowledge_source_name=name,
        include_references=True,
        include_reference_source_data=True,
    )
    for name in KNOWLEDGE_SOURCE_NAMES.values()
]

question = (
    "Why did the Ghana bars sold at Larkspur Market turn grey and dusty-looking in July 2026, "
    "and was the batch itself bad?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=knowledge_source_params,
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

Look for `searchIndex` activities. Which of the four sources did the knowledge base query, and what queries did it send to each? A mystery question usually needs several sources. The combined KB does not include the standalone file source.


In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

References identify which document supports each claim. The combined KB's indexed references have `type: "searchIndex"` and include `title`, `blob_path`, and page numbers, so you can open the PDF in `data/cocoarynth/documents/` and check the evidence yourself.


In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### Source hunt

Review the references and activity log for the grey Ghana bars:

1. Which document is the retailer's complaint, and which batch code does it mention?
2. Which carrier record shows where the pallet waited, and for how long?
3. How hot did it get, and which document shows it?
4. What did the factory find when it checked bars it had kept from the same batch?
5. Which policy or guide explains what heat does to chocolate?
6. Do the references expose `title`, `blob_path`, `page_number_from`, and `page_number_to`?

Some evidence only appears in charts, photos, or scanned forms. If an answer seems to be missing a detail, open the cited PDF and look.


## Bonus: Query from the GitHub Copilot App

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or the GitHub Copilot App. Run the cell below to print the MCP URL for the Cocoarynth knowledge base, then follow the steps after it to query the knowledge base from the GitHub Copilot App.

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(mcp_url)

### Sign in to GitHub

Sign in through the [Skillable Events GitHub enterprise](https://github.com/enterprises/skillable-events/sso).

You will be signed into a special account created for the lab environment, not your actual GitHub account.

### Start the GitHub Copilot App

1. Open the GitHub Copilot App by finding the shortcut on the desktop.
2. Select "Sign in to GitHub" button.
3. When the browser pops up, paste the device code that was automatically copied into your clipboard, and select "Authorize github".
4. Return to the app. You don't need to connect any repositories, so select "Continue" on repository selection screen and "Finish" on the preferences screen.


### Add the knowledge base MCP server

1. Select **Customize** in the sidebar.
2. Select the **MCP** tab.
3. Open the **Add** menu, then select **MCP server**.
4. Fill in the details for the Cocoarynth knowledge base:
   * Server name: `cocoarynth-kb`
   * Server type: **HTTP** (*not* Local/SSE)
   * URL: Paste the URL printed by the cell above.
   * Headers: Enter `api-key` for the header name and the value of `AZURE_SEARCH_ADMIN_KEY` from your `.env` file for its value.
5. Leave other fields at their defaults, then select **Add server**.
6. Confirm that `cocoarynth-kb` appears in the list of MCP servers.

### Ask a question grounded by the knowledge base

Create a new chat by selecting **+** next to **Chats** in the sidebar, then ask one of the mysteries you haven't solved yet. For example:

_"Which origin's beans cost Cocoarynth the most in 2026, and was it worth paying for them, judging by taste and sales?"_


## Mission complete

**What you built:**

* **File knowledge source**: Cocoarynth's label and marketing claims policy uploaded directly for questions about wrapper claims.
* **Indexed knowledge sources**: Four indexes of origin, quality, logistics and retail, and policy records with portable citation metadata.
* **Multi-source knowledge base**: `cocoarynth-kb-all`, combining the four indexed sources to answer questions whose evidence is spread across farms, the factory, carriers, retailers, and policies.
